# 🎬 Agente de YouTube — versión para el navegador

Ejecuta las celdas **de arriba hacia abajo**. En cada una, pulsa el botón **▶️** (a la izquierda) y espera a que termine antes de pasar a la siguiente.

1. Instalar → 2. Cargar el código → 3. Pegar tu clave → 4. Ejecutar → 5. Descargar resultados.


### 1) Instalar lo necesario (tarda ~30 seg)

In [ ]:
!pip install -q anthropic requests
print('Listo ✅')

### 2) Cargar el código del agente

In [ ]:
import os
os.makedirs('src', exist_ok=True)
os.makedirs('salidas', exist_ok=True)
print('Carpetas creadas ✅')

In [ ]:
%%writefile src/__init__.py


In [ ]:
%%writefile src/config.py
"""Configuración central del agente. Lee variables desde .env o el entorno."""
from __future__ import annotations

import os
from dataclasses import dataclass

try:
    from dotenv import load_dotenv

    load_dotenv()
except ImportError:  # dotenv es opcional; el entorno igual funciona
    pass


@dataclass
class Config:
    anthropic_api_key: str | None
    claude_model: str
    idioma: str
    youtube_api_key: str | None
    pexels_api_key: str | None

    @property
    def tiene_youtube(self) -> bool:
        return bool(self.youtube_api_key)

    @property
    def tiene_pexels(self) -> bool:
        return bool(self.pexels_api_key)


def cargar_config() -> Config:
    return Config(
        # anthropic_api_key puede ser None: el SDK también resuelve credenciales
        # desde un perfil `ant auth login`. Solo lo usamos para avisar al usuario.
        anthropic_api_key=os.getenv("ANTHROPIC_API_KEY"),
        claude_model=os.getenv("CLAUDE_MODEL", "claude-opus-5"),
        idioma=os.getenv("IDIOMA", "es"),
        youtube_api_key=os.getenv("YOUTUBE_API_KEY") or None,
        pexels_api_key=os.getenv("PEXELS_API_KEY") or None,
    )


In [ ]:
%%writefile src/claude_client.py
"""Envoltorio ligero sobre el SDK de Anthropic.

Centraliza las llamadas a Claude para el resto del agente: texto libre,
salida JSON robusta y búsqueda web (con la herramienta server-side de Claude).
"""
from __future__ import annotations

import json
import re

import anthropic

from .config import Config


class ClaudeClient:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        # El SDK resuelve la credencial desde ANTHROPIC_API_KEY o un perfil de `ant`.
        self.client = anthropic.Anthropic()
        self.model = cfg.claude_model

    # ── Texto libre (streaming para salidas largas como el guion) ───────────
    def texto(self, prompt: str, system: str = "", max_tokens: int = 16000) -> str:
        partes: list[str] = []
        with self.client.messages.stream(
            model=self.model,
            max_tokens=max_tokens,
            system=system or anthropic.NOT_GIVEN,
            messages=[{"role": "user", "content": prompt}],
        ) as stream:
            for texto in stream.text_stream:
                partes.append(texto)
        return "".join(partes).strip()

    # ── Salida JSON robusta ─────────────────────────────────────────────────
    def json(self, prompt: str, system: str = "", max_tokens: int = 8000) -> dict | list:
        instruccion = (
            "\n\nResponde ÚNICAMENTE con JSON válido, sin texto adicional, "
            "sin bloques de código markdown."
        )
        crudo = self.texto(prompt + instruccion, system=system, max_tokens=max_tokens)
        return _extraer_json(crudo)

    # ── Investigación con búsqueda web ──────────────────────────────────────
    def investigar_web(self, prompt: str, system: str = "", max_tokens: int = 12000) -> str:
        """Usa la herramienta de búsqueda web de Claude para datos actuales."""
        resp = self.client.messages.create(
            model=self.model,
            max_tokens=max_tokens,
            system=system or anthropic.NOT_GIVEN,
            tools=[{"type": "web_search_20260209", "name": "web_search", "max_uses": 8}],
            messages=[{"role": "user", "content": prompt}],
        )
        # Reanudar si el bucle server-side se pausó (pause_turn).
        mensajes = [{"role": "user", "content": prompt}]
        vueltas = 0
        while resp.stop_reason == "pause_turn" and vueltas < 5:
            mensajes = [
                {"role": "user", "content": prompt},
                {"role": "assistant", "content": resp.content},
            ]
            resp = self.client.messages.create(
                model=self.model,
                max_tokens=max_tokens,
                system=system or anthropic.NOT_GIVEN,
                tools=[{"type": "web_search_20260209", "name": "web_search", "max_uses": 8}],
                messages=mensajes,
            )
            vueltas += 1

        return "".join(b.text for b in resp.content if b.type == "text").strip()


def _extraer_json(crudo: str) -> dict | list:
    """Parsea JSON tolerando envoltorios de markdown o texto alrededor."""
    crudo = crudo.strip()
    # Quitar cercas de código ```json ... ```
    crudo = re.sub(r"^```(?:json)?\s*", "", crudo)
    crudo = re.sub(r"\s*```$", "", crudo)
    try:
        return json.loads(crudo)
    except json.JSONDecodeError:
        pass
    # Buscar el primer objeto/arreglo balanceado
    for apertura, cierre in (("{", "}"), ("[", "]")):
        inicio = crudo.find(apertura)
        fin = crudo.rfind(cierre)
        if inicio != -1 and fin != -1 and fin > inicio:
            try:
                return json.loads(crudo[inicio : fin + 1])
            except json.JSONDecodeError:
                continue
    raise ValueError(f"No se pudo parsear JSON de la respuesta:\n{crudo[:500]}")


In [ ]:
%%writefile src/youtube_research.py
"""Etapa 1: investigación de nicho.

Con clave de YouTube: datos reales (vistas, suscriptores, "Viral Score").
Sin clave: respaldo con búsqueda web de Claude.
"""
from __future__ import annotations

from datetime import datetime, timedelta, timezone

import requests

from .claude_client import ClaudeClient
from .config import Config

_YT = "https://www.googleapis.com/youtube/v3"

# Umbrales para detectar "canales joya": pocos subs, muchas vistas.
MAX_SUBS_JOYA = 50_000
MIN_VISTAS_JOYA = 50_000


def investigar(cfg: Config, claude: ClaudeClient, nicho: str) -> dict:
    """Devuelve un informe de investigación del nicho."""
    print(f"🔎 Investigando el nicho: «{nicho}» ...")

    # Paso A: pedir a Claude palabras clave y ángulos semilla (siempre).
    semillas = _keywords_semilla(claude, nicho)

    if cfg.tiene_youtube:
        print("   → Usando la API oficial de YouTube (datos reales).")
        datos_yt = _investigar_youtube(cfg.youtube_api_key, semillas["consultas"])
    else:
        print("   → Sin YOUTUBE_API_KEY: usando búsqueda web de Claude (respaldo).")
        datos_yt = _investigar_web(claude, nicho)

    return {
        "nicho": nicho,
        "palabras_clave": semillas["palabras_clave"],
        "angulos_video": semillas["angulos_video"],
        "consultas_busqueda": semillas["consultas"],
        **datos_yt,
    }


def _keywords_semilla(claude: ClaudeClient, nicho: str) -> dict:
    prompt = f"""Eres un experto en SEO y estrategia de YouTube.
Para el nicho: "{nicho}", genera un plan de investigación de palabras clave.

Devuelve JSON con esta forma exacta:
{{
  "palabras_clave": ["12-18 palabras clave con intención de búsqueda real en YouTube"],
  "angulos_video": ["8-10 ángulos/ideas de video con alto potencial de clic (títulos tentativos)"],
  "consultas": ["4-6 consultas de búsqueda para explorar la competencia en YouTube"]
}}"""
    data = claude.json(prompt)
    return {
        "palabras_clave": data.get("palabras_clave", []),
        "angulos_video": data.get("angulos_video", []),
        "consultas": data.get("consultas", [])[:6],
    }


# ── Camino A: API de YouTube ────────────────────────────────────────────────
def _investigar_youtube(api_key: str, consultas: list[str]) -> dict:
    publicado_despues = (
        datetime.now(timezone.utc) - timedelta(days=180)
    ).isoformat().replace("+00:00", "Z")

    video_ids: list[str] = []
    origen: dict[str, str] = {}  # video_id -> consulta que lo encontró
    for consulta in consultas[:4]:  # limitar cuota
        for orden in ("relevance", "viewCount"):
            try:
                r = requests.get(
                    f"{_YT}/search",
                    params={
                        "key": api_key,
                        "q": consulta,
                        "part": "snippet",
                        "type": "video",
                        "order": orden,
                        "maxResults": 10,
                        "publishedAfter": publicado_despues,
                        "relevanceLanguage": "es",
                    },
                    timeout=30,
                )
                r.raise_for_status()
            except requests.RequestException as e:
                print(f"   ⚠️  Error en búsqueda de YouTube ({consulta}/{orden}): {e}")
                continue
            for item in r.json().get("items", []):
                vid = item["id"].get("videoId")
                if vid and vid not in origen:
                    video_ids.append(vid)
                    origen[vid] = consulta

    if not video_ids:
        return {"tendencias": [], "competencia": [], "canales_joya": []}

    videos = _detalles_videos(api_key, video_ids)
    canal_ids = list({v["channel_id"] for v in videos if v.get("channel_id")})
    subs = _suscriptores_canales(api_key, canal_ids)

    competencia = []
    for v in videos:
        s = subs.get(v["channel_id"], 0)
        vistas = v["vistas"]
        ratio = round(vistas / max(s, 1), 2)
        competencia.append(
            {
                "titulo": v["titulo"],
                "canal": v["canal"],
                "vistas": vistas,
                "suscriptores": s,
                "viral_score": ratio,  # vistas por suscriptor
                "url": f"https://youtu.be/{v['id']}",
                "consulta": origen.get(v["id"], ""),
            }
        )

    competencia.sort(key=lambda x: x["vistas"], reverse=True)

    # Canales joya: pocos subs, muchas vistas → mejor Viral Score.
    joyas = [
        c
        for c in competencia
        if c["suscriptores"] <= MAX_SUBS_JOYA and c["vistas"] >= MIN_VISTAS_JOYA
    ]
    joyas.sort(key=lambda x: x["viral_score"], reverse=True)

    return {
        "tendencias": [c["titulo"] for c in competencia[:10]],
        "competencia": competencia[:20],
        "canales_joya": joyas[:10],
    }


def _detalles_videos(api_key: str, video_ids: list[str]) -> list[dict]:
    out: list[dict] = []
    for i in range(0, len(video_ids), 50):
        lote = video_ids[i : i + 50]
        r = requests.get(
            f"{_YT}/videos",
            params={
                "key": api_key,
                "id": ",".join(lote),
                "part": "snippet,statistics",
            },
            timeout=30,
        )
        r.raise_for_status()
        for item in r.json().get("items", []):
            stats = item.get("statistics", {})
            snip = item.get("snippet", {})
            out.append(
                {
                    "id": item["id"],
                    "titulo": snip.get("title", ""),
                    "canal": snip.get("channelTitle", ""),
                    "channel_id": snip.get("channelId", ""),
                    "vistas": int(stats.get("viewCount", 0)),
                }
            )
    return out


def _suscriptores_canales(api_key: str, canal_ids: list[str]) -> dict[str, int]:
    subs: dict[str, int] = {}
    for i in range(0, len(canal_ids), 50):
        lote = canal_ids[i : i + 50]
        r = requests.get(
            f"{_YT}/channels",
            params={"key": api_key, "id": ",".join(lote), "part": "statistics"},
            timeout=30,
        )
        r.raise_for_status()
        for item in r.json().get("items", []):
            stats = item.get("statistics", {})
            oculto = stats.get("hiddenSubscriberCount", False)
            subs[item["id"]] = 0 if oculto else int(stats.get("subscriberCount", 0))
    return subs


# ── Camino B: respaldo con búsqueda web de Claude ────────────────────────────
def _investigar_web(claude: ClaudeClient, nicho: str) -> dict:
    prompt = f"""Investiga en la web el panorama actual de YouTube para el nicho: "{nicho}".
Busca videos recientes populares, tendencias y canales que estén creciendo.

Devuelve JSON con esta forma exacta (usa datos reales que encuentres; si un dato
no está disponible pon null):
{{
  "tendencias": ["8-10 temas/formatos que están funcionando ahora en este nicho"],
  "competencia": [
    {{"titulo": "...", "canal": "...", "vistas": 123456, "suscriptores": null,
      "viral_score": null, "url": "...", "consulta": ""}}
  ],
  "canales_joya": [
    {{"titulo": "video que despegó", "canal": "canal pequeño", "vistas": 200000,
      "suscriptores": 5000, "viral_score": 40.0, "url": "...", "consulta": ""}}
  ]
}}
Prioriza en "canales_joya" canales con pocos suscriptores pero muchas vistas."""
    try:
        crudo = claude.investigar_web(prompt)
        from .claude_client import _extraer_json

        data = _extraer_json(crudo)
    except Exception as e:  # noqa: BLE001
        print(f"   ⚠️  Falló la investigación web: {e}")
        return {"tendencias": [], "competencia": [], "canales_joya": []}
    return {
        "tendencias": data.get("tendencias", []),
        "competencia": data.get("competencia", []),
        "canales_joya": data.get("canales_joya", []),
    }


In [ ]:
%%writefile src/script_generator.py
"""Etapa 2: guion optimizado + paquete SEO, listo para copiar y pegar."""
from __future__ import annotations

from .claude_client import ClaudeClient

_SISTEMA = (
    "Eres un guionista experto en YouTube y en el algoritmo de la plataforma. "
    "Dominas hooks de retención, ritmo, storytelling y CTAs. Escribes guiones "
    "que la gente termina de ver y que YouTube recomienda."
)


def generar_guion(
    claude: ClaudeClient,
    investigacion: dict,
    tema: str,
    duracion_min: int = 8,
    idioma: str = "es",
) -> dict:
    print(f"✍️  Generando guion optimizado para: «{tema}» ({duracion_min} min) ...")

    contexto = _contexto_investigacion(investigacion)

    # 1) Guion completo como texto (llamada dedicada para máxima calidad).
    prompt_guion = f"""Idioma de salida: {idioma}.

Escribe un GUION COMPLETO de YouTube, listo para copiar y pegar y grabar tal cual.

Tema del video: "{tema}"
Duración objetivo: ~{duracion_min} minutos.

Contexto de investigación del nicho (úsalo para acertar el ángulo y las palabras clave):
{contexto}

Requisitos del guion:
- HOOK potente en los primeros 5-10 segundos (evita introducciones lentas).
- Estructura clara con secciones marcadas: [GANCHO], [INTRO], [DESARROLLO 1..N], [CLÍMAX], [CTA/CIERRE].
- Marca visualmente dónde va cada corte de escena con la etiqueta «[ESCENA n]» al inicio de cada bloque visual.
- Lenguaje natural y hablado (como se dice en voz alta), frases cortas.
- Incluye indicaciones breves de b-roll/entonación entre paréntesis cuando ayude.
- Optimizado para retención: bucles abiertos, promesas que se cumplen, transiciones ágiles.
- Termina con un CTA claro (suscripción + siguiente video).

Devuelve SOLO el guion (sin comentarios previos)."""
    guion = claude.texto(prompt_guion, system=_SISTEMA, max_tokens=20000)

    # 2) Paquete SEO + lista de escenas (JSON) a partir del guion ya escrito.
    prompt_seo = f"""Idioma de salida: {idioma}.

A partir de este guion de YouTube, genera el paquete de publicación y la lista
de escenas para imágenes.

GUION:
\"\"\"
{guion}
\"\"\"

Contexto de palabras clave del nicho: {", ".join(investigacion.get("palabras_clave", [])[:15])}

Devuelve JSON con esta forma exacta:
{{
  "titulos": ["5 títulos optimizados para CTR, con curiosidad/beneficio, <70 caracteres"],
  "titulo_recomendado": "el mejor de los 5",
  "descripcion": "descripción de YouTube (2-4 párrafos) con palabras clave naturales y un resumen; incluye 1-2 líneas de gancho al inicio",
  "etiquetas": ["15-25 tags relevantes en minúscula"],
  "hashtags": ["3-5 hashtags con # incluido"],
  "texto_miniatura": ["2-3 opciones de texto MUY corto (2-4 palabras) para la miniatura"],
  "capitulos": ["timestamps sugeridos, formato '0:00 Título'"],
  "escenas": [
    {{"n": 1, "resumen": "qué se ve/dice en la escena", "palabras_clave_visuales": "términos concretos para buscar/generar la imagen"}}
  ]
}}
Extrae UNA escena por cada bloque [ESCENA n] del guion (o divide en 6-12 escenas lógicas si no hay etiquetas)."""
    seo = claude.json(prompt_seo, system=_SISTEMA, max_tokens=12000)

    return {"tema": tema, "duracion_min": duracion_min, "guion": guion, **seo}


def _contexto_investigacion(inv: dict) -> str:
    lineas = []
    if inv.get("palabras_clave"):
        lineas.append("Palabras clave: " + ", ".join(inv["palabras_clave"][:15]))
    if inv.get("tendencias"):
        lineas.append("Tendencias/formatos actuales: " + "; ".join(inv["tendencias"][:8]))
    if inv.get("canales_joya"):
        joyas = "; ".join(
            f"{c.get('titulo','')} ({c.get('vistas','?')} vistas)"
            for c in inv["canales_joya"][:5]
        )
        lineas.append("Videos de canales pequeños que explotaron: " + joyas)
    return "\n".join(lineas) or "(sin datos de investigación)"


In [ ]:
%%writefile src/image_prompts.py
"""Etapa 3: prompts de imágenes para cada escena del guion.

Genera un prompt detallado (para Midjourney/DALL·E/Stable Diffusion) y una
consulta de búsqueda para banco de imágenes por cada escena.
"""
from __future__ import annotations

import json

from .claude_client import ClaudeClient

_SISTEMA = (
    "Eres director de arte especializado en imágenes para YouTube. Escribes "
    "prompts de generación de imágenes ricos en detalle (composición, estilo, "
    "iluminación, encuadre, cámara) y consultas de banco de imágenes efectivas."
)


def generar_prompts_imagenes(
    claude: ClaudeClient, escenas: list[dict], idioma: str = "es"
) -> list[dict]:
    if not escenas:
        return []
    print(f"🎨 Generando prompts de imágenes para {len(escenas)} escenas ...")

    escenas_txt = json.dumps(escenas, ensure_ascii=False, indent=2)
    prompt = f"""Idioma de las descripciones: {idioma}.

Para cada escena, crea material visual. Devuelve JSON: un arreglo donde cada
elemento tenga esta forma exacta:
{{
  "n": <número de escena>,
  "prompt_imagen": "prompt DETALLADO en inglés para generadores de imágenes (Midjourney/DALL·E/SD): sujeto, acción, entorno, estilo, iluminación, encuadre, lente, mood; sin texto dentro de la imagen",
  "prompt_imagen_es": "el mismo concepto explicado brevemente en {idioma}",
  "consulta_stock": "2-4 palabras en inglés, concretas y visuales, para buscar en bancos de imágenes (Pexels)"
}}

Escenas:
{escenas_txt}

Devuelve SOLO el arreglo JSON, en el mismo orden."""
    data = claude.json(prompt, system=_SISTEMA, max_tokens=12000)
    if isinstance(data, dict):  # por si el modelo envuelve en una clave
        for v in data.values():
            if isinstance(v, list):
                data = v
                break
    return data if isinstance(data, list) else []


In [ ]:
%%writefile src/agent.py
"""Agente experto en YouTube: investigación → guion → prompts → imágenes.

Uso:
    python -m src.agent "nicho o tema del canal"
    python -m src.agent "finanzas personales" --tema "3 errores al invertir" --duracion 10
"""
from __future__ import annotations

import argparse
import json
import re
import sys
from datetime import datetime
from pathlib import Path

from .claude_client import ClaudeClient
from .config import Config, cargar_config
from .image_prompts import generar_prompts_imagenes
from .script_generator import generar_guion
from .stock_images import descargar_imagenes
from .youtube_research import investigar

RAIZ = Path(__file__).resolve().parent.parent
SALIDAS = RAIZ / "salidas"


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(
        description="Agente de IA experto en YouTube (investigación, guion, imágenes)."
    )
    parser.add_argument("nicho", help="Nicho o tema del canal, p. ej. 'finanzas personales'")
    parser.add_argument("--tema", default=None, help="Tema específico del video (opcional)")
    parser.add_argument("--duracion", type=int, default=8, help="Duración en minutos (def. 8)")
    parser.add_argument(
        "--imagenes-por-escena", type=int, default=1, help="Fotos de stock por escena (def. 1)"
    )
    parser.add_argument(
        "--solo-investigacion", action="store_true", help="Solo hacer la investigación"
    )
    args = parser.parse_args(argv)

    cfg = cargar_config()
    _avisos_config(cfg)

    try:
        claude = ClaudeClient(cfg)
    except Exception as e:  # noqa: BLE001
        print(f"❌ No se pudo inicializar Claude: {e}")
        print("   Configura ANTHROPIC_API_KEY (copia .env.example a .env).")
        return 1

    try:
        return _pipeline(cfg, claude, args)
    except Exception as e:  # noqa: BLE001
        return _manejar_error(e)


def _pipeline(cfg: Config, claude: ClaudeClient, args) -> int:
    # 1) Investigación
    inv = investigar(cfg, claude, args.nicho)

    if args.solo_investigacion:
        carpeta = _preparar_carpeta(args.nicho)
        _escribir_investigacion(carpeta, inv)
        (carpeta / "proyecto.json").write_text(
            json.dumps({"investigacion": inv}, ensure_ascii=False, indent=2), encoding="utf-8"
        )
        print(f"\n✅ Investigación lista en: {carpeta}")
        return 0

    # 2) Elegir tema del video
    tema = args.tema or _elegir_tema(claude, inv)
    print(f"🎯 Tema del video: «{tema}»")

    # 3) Guion + SEO
    paquete = generar_guion(claude, inv, tema, args.duracion, cfg.idioma)

    # 4) Prompts de imágenes
    prompts = generar_prompts_imagenes(claude, paquete.get("escenas", []), cfg.idioma)

    # 5) Descargar imágenes de stock
    carpeta = _preparar_carpeta(tema)
    imagenes = descargar_imagenes(
        cfg.pexels_api_key, prompts, carpeta, args.imagenes_por_escena
    )

    # 6) Escribir paquete de entrega
    _escribir_investigacion(carpeta, inv)
    _escribir_guion(carpeta, paquete)
    _escribir_prompts(carpeta, prompts, imagenes)
    (carpeta / "proyecto.json").write_text(
        json.dumps(
            {
                "investigacion": inv,
                "paquete": paquete,
                "prompts_imagenes": prompts,
                "imagenes": imagenes,
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    print(f"\n✅ ¡Listo! Paquete completo en: {carpeta}")
    print("   • guion.md            → guion + SEO para copiar y pegar")
    print("   • prompts_imagenes.md → prompts + consultas de imágenes")
    print("   • investigacion.md    → competencia, tendencias y canales joya")
    print("   • imagenes/           → fotos de stock descargadas")
    return 0


def _manejar_error(e: Exception) -> int:
    msg = str(e)
    es_auth = (
        "authentication" in msg.lower()
        or "api_key" in msg.lower()
        or "x-api-key" in msg.lower()
        or "401" in msg
    )
    if es_auth:
        print("\n❌ No hay credencial de Claude válida.")
        print("   Configura ANTHROPIC_API_KEY: copia .env.example a .env y pega tu clave")
        print("   (la obtienes en https://console.anthropic.com).")
    else:
        print(f"\n❌ Error durante el proceso: {e}")
    return 1


# ── Selección de tema ────────────────────────────────────────────────────────
def _elegir_tema(claude: ClaudeClient, inv: dict) -> str:
    angulos = inv.get("angulos_video", [])
    if not angulos:
        return inv.get("nicho", "Video de YouTube")
    prompt = f"""Del siguiente listado de ideas de video para el nicho "{inv.get('nicho')}",
elige la de MAYOR potencial (CTR + retención + tendencia actual) y devuélvela
como un único título de video, sin explicaciones.

Tendencias actuales: {"; ".join(inv.get("tendencias", [])[:8])}

Ideas:
{chr(10).join(f"- {a}" for a in angulos)}"""
    try:
        elegido = claude.texto(prompt, max_tokens=200).strip().strip('"')
        return elegido or angulos[0]
    except Exception:  # noqa: BLE001
        return angulos[0]


# ── Escritura de archivos ──────────────────────────────────────────────────
def _preparar_carpeta(nombre: str) -> Path:
    slug = _slug(nombre)
    ts = datetime.now().strftime("%Y%m%d-%H%M%S")
    carpeta = SALIDAS / f"{slug}-{ts}"
    carpeta.mkdir(parents=True, exist_ok=True)
    return carpeta


def _escribir_investigacion(carpeta: Path, inv: dict) -> None:
    L = [f"# Investigación de nicho: {inv.get('nicho','')}\n"]

    L.append("## Palabras clave\n")
    L += [f"- {k}" for k in inv.get("palabras_clave", [])]

    L.append("\n## Ideas de video (ángulos)\n")
    L += [f"- {a}" for a in inv.get("angulos_video", [])]

    L.append("\n## Tendencias / formatos actuales\n")
    L += [f"- {t}" for t in inv.get("tendencias", [])]

    joyas = inv.get("canales_joya", [])
    L.append("\n## 💎 Canales joya (pocos subs, muchas vistas)\n")
    if joyas:
        L.append("| Video | Canal | Vistas | Subs | Viral Score | Enlace |")
        L.append("|---|---|---:|---:|---:|---|")
        for c in joyas:
            L.append(
                f"| {c.get('titulo','')} | {c.get('canal','')} | {_num(c.get('vistas'))} "
                f"| {_num(c.get('suscriptores'))} | {c.get('viral_score','')} | {c.get('url','')} |"
            )
    else:
        L.append("_No se detectaron canales joya con los datos disponibles._")

    comp = inv.get("competencia", [])
    L.append("\n## Competencia (top por vistas)\n")
    if comp:
        L.append("| Video | Canal | Vistas | Subs | Viral Score | Enlace |")
        L.append("|---|---|---:|---:|---:|---|")
        for c in comp:
            L.append(
                f"| {c.get('titulo','')} | {c.get('canal','')} | {_num(c.get('vistas'))} "
                f"| {_num(c.get('suscriptores'))} | {c.get('viral_score','')} | {c.get('url','')} |"
            )
    else:
        L.append("_Sin datos de competencia._")

    (carpeta / "investigacion.md").write_text("\n".join(L) + "\n", encoding="utf-8")


def _escribir_guion(carpeta: Path, p: dict) -> None:
    L = [f"# Guion: {p.get('tema','')}\n"]

    L.append("## 🏷️ Título recomendado\n")
    L.append(f"**{p.get('titulo_recomendado','')}**\n")
    if p.get("titulos"):
        L.append("### Otras opciones de título\n")
        L += [f"- {t}" for t in p["titulos"]]

    if p.get("texto_miniatura"):
        L.append("\n## 🖼️ Texto para la miniatura\n")
        L += [f"- {t}" for t in p["texto_miniatura"]]

    L.append("\n## 🎬 GUION (copiar y pegar)\n")
    L.append(p.get("guion", ""))

    if p.get("capitulos"):
        L.append("\n## ⏱️ Capítulos (timestamps)\n")
        L += [f"{c}" for c in p["capitulos"]]

    L.append("\n## 📝 Descripción de YouTube\n")
    L.append(p.get("descripcion", ""))

    if p.get("hashtags"):
        L.append("\n## Hashtags\n")
        L.append(" ".join(p["hashtags"]))

    if p.get("etiquetas"):
        L.append("\n## 🏷️ Etiquetas (tags)\n")
        L.append(", ".join(p["etiquetas"]))

    (carpeta / "guion.md").write_text("\n".join(L) + "\n", encoding="utf-8")


def _escribir_prompts(carpeta: Path, prompts: list[dict], imagenes: list[dict]) -> None:
    img_por_escena: dict = {}
    for im in imagenes:
        img_por_escena.setdefault(im["escena"], []).append(im)

    L = ["# Prompts de imágenes por escena\n"]
    for p in prompts:
        n = p.get("n", "?")
        L.append(f"## Escena {n}\n")
        if p.get("prompt_imagen_es"):
            L.append(f"**Idea:** {p['prompt_imagen_es']}\n")
        L.append("**Prompt (generadores de imagen):**\n")
        L.append(f"```\n{p.get('prompt_imagen','')}\n```\n")
        L.append(f"**Consulta de banco de imágenes:** `{p.get('consulta_stock','')}`\n")
        for im in img_por_escena.get(n, []):
            L.append(f"- 📷 `{im['archivo']}` — foto de {im.get('autor','')} ({im.get('credito_url','')})")
        L.append("")

    (carpeta / "prompts_imagenes.md").write_text("\n".join(L) + "\n", encoding="utf-8")


# ── Utilidades ───────────────────────────────────────────────────────────────
def _slug(texto: str) -> str:
    s = re.sub(r"[^\w\s-]", "", texto.lower())
    s = re.sub(r"[\s_-]+", "-", s).strip("-")
    return (s or "video")[:50]


def _num(v) -> str:
    try:
        return f"{int(v):,}".replace(",", ".")
    except (TypeError, ValueError):
        return str(v) if v is not None else "?"


def _avisos_config(cfg: Config) -> None:
    if not cfg.anthropic_api_key:
        print("ℹ️  ANTHROPIC_API_KEY no está en el entorno; se intentará usar el perfil de `ant`.")
    if not cfg.tiene_youtube:
        print("ℹ️  Sin YOUTUBE_API_KEY: la investigación usará búsqueda web (menos preciso en subs/vistas).")
    if not cfg.tiene_pexels:
        print("ℹ️  Sin PEXELS_API_KEY: se generarán prompts de imágenes pero no se descargarán fotos.")


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
print('Código del agente cargado ✅')

### 3) Pega tus claves
Al ejecutar, aparecerá un recuadro. Pega tu clave de **Anthropic** (obligatoria) y pulsa Enter. Las otras dos son opcionales: si no las tienes, deja el recuadro vacío y pulsa Enter.

In [ ]:
from getpass import getpass
import os

clave = getpass('Clave de Anthropic (empieza con sk-ant-): ').strip()
if not clave:
    raise SystemExit('❌ Necesitas la clave de Anthropic para continuar.')
os.environ['ANTHROPIC_API_KEY'] = clave

yt = getpass('YOUTUBE_API_KEY (opcional, Enter para omitir): ').strip()
if yt: os.environ['YOUTUBE_API_KEY'] = yt

px = getpass('PEXELS_API_KEY (opcional, Enter para omitir): ').strip()
if px: os.environ['PEXELS_API_KEY'] = px

print('Claves guardadas para esta sesión ✅')

### 4) Escribe tu nicho y ejecuta
Cambia los valores del formulario de la derecha y pulsa ▶️. Puede tardar 1–3 minutos.

In [ ]:
#@title ▶️ Ejecutar el agente { display-mode: "form" }
nicho = "finanzas personales"  #@param {type:"string"}
tema = ""  #@param {type:"string"}
duracion = 8  #@param {type:"integer"}

import sys, subprocess
args = [sys.executable, '-m', 'src.agent', nicho, '--duracion', str(duracion)]
if tema.strip():
    args += ['--tema', tema.strip()]
print('⏳ Trabajando... (investigación → guion → prompts → imágenes)\n')
subprocess.run(args)

### 5) Ver el guion y descargar todo

In [ ]:
#@title 📄 Mostrar el guion y descargar el paquete (.zip)
import glob, os, shutil
from IPython.display import Markdown, display
from google.colab import files

carpetas = sorted(glob.glob('salidas/*/'), key=os.path.getmtime)
if not carpetas:
    print('Aún no hay resultados. Ejecuta la celda 4 primero.')
else:
    ultima = carpetas[-1].rstrip('/')
    print('📂 Resultados en:', ultima, '\n')
    guion = os.path.join(ultima, 'guion.md')
    if os.path.exists(guion):
        with open(guion, encoding='utf-8') as f:
            display(Markdown(f.read()))
    shutil.make_archive(ultima, 'zip', ultima)
    print('\n⬇️ Descargando el paquete completo...')
    files.download(ultima + '.zip')